In [1]:
from the_well.data.normalization import ZScoreNormalization
from pathlib import Path
import numpy as np
import pickle
import fsspec
import torch
import yaml
import sys
import re

PROJECT_ROOT = "/Users/gabrieleaccarino/Workspace/waveletflux"
sys.path.insert(0, PROJECT_ROOT)

from waveletflux.utils.well_data import get_well_dataset, get_well_inputs
from waveletflux.modules.latent_fm import LatentFlowModule

print("Imports OK | MPS:", torch.backends.mps.is_available())
print(f"MPS allocated : {torch.mps.current_allocated_memory() / 1e9:.2f} GB")
print(f"MPS reserved  : {torch.mps.driver_allocated_memory()  / 1e9:.2f} GB")

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

Imports OK | MPS: True
MPS allocated : 0.00 GB
MPS reserved  : 0.00 GB


In [2]:
#CONFIG_PATH = f"{PROJECT_ROOT}/configs/turbulent_radiative_layer_2D/WaveletFlow_turbulent_radiative_layer_2D_3scales_db2_v0.0.toml"
well_base_path    = f"/Users/gabrieleaccarino/Workspace/multi-scale-wavelet-flow-matching/data/datasets"
well_dataset_name = 'turbulent_radiative_layer_2D'
stats_path        = f"{well_base_path}/{well_dataset_name}/stats.yaml"

ckpt_name         = "LatentFlow_turbulent_radiative_layer_2D_last_v1.0_20260327_170226-epoch=87-val_loss=0.1248_portable.ckpt"

ckpt_base_name    = re.split(r"_\d{8}_", ckpt_name)[0]
ckpt_path         = f"{PROJECT_ROOT}/experiments/turbulent_radiative_layer_2D/ckpts/{ckpt_name}"

In [3]:
with fsspec.open(stats_path, mode='r') as f:
    stats = yaml.safe_load(f)

model = LatentFlowModule.from_checkpoint(ckpt_path, map_location=device)
model.eval()
model.to(device)
print("Model loaded from checkpoint.")

/Users/gabrieleaccarino/Workspace/waveletflux/waveletflux/modules/latent_fm.py:595: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location=map_lo

Model loaded from checkpoint.


In [4]:
dataset = get_well_dataset(
    path    = well_base_path,
    physics = well_dataset_name,
    split   = 'test',
    steps   = -1,     # load the full trajectory
)

In [5]:
N_TRAJ             = np.arange(len(dataset.metadata.n_steps_per_trajectory)) 
N_ROLLOUT_STEPS    = dataset.metadata.n_steps_per_trajectory[0]                         # autoregressive steps after the seed window
N_STEPS_INPUT      = 4                                                                  # Ti — sliding-window width (must match training)
NUM_SAMPLING_STEPS = 20                                                                 # ODE steps per prediction
N_ENSEMBLE         = 1                                                                 # flow-matching ensemble size
#device_ids         = [device_idx for device_idx in range(torch.cuda.device_count())]    # GPU device indices for DataParallel

In [6]:
pred_trajs = []
real_trajs = []

for traj_idx in N_TRAJ:
    print(f"\n{'━'*60}")
    print(f"  Trajectory {traj_idx}")
    print(f"{'━'*60}")

    item          = dataset[traj_idx]
    x_full, label = get_well_inputs(item, device=device)      # (N_ROLLOUT_STEPS, H, W, C), physical

    preds = model.rollout(
        seed          = x_full[:N_STEPS_INPUT].unsqueeze(0),  # (1, N_STEPS_INPUT, H, W, C), physical
        scalar_conds  = label.unsqueeze(0),                   # (1, num_cond), scalar conditions
        rollout_steps = N_ROLLOUT_STEPS,
        n_ensemble    = N_ENSEMBLE,
        verbose       = True,
    )
    # preds: (1, N_ENSEMBLE, N_ROLLOUT_STEPS, H, W, C)

    pred_trajs.append(preds.detach().cpu())
    real_trajs.append(x_full.unsqueeze(0).detach().cpu())


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 0
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  rollout call  101/101 (frames 101–101/101)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 1
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  rollout call  101/101 (frames 101–101/101)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  rollout call  101/101 (frames 101–101/101)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 3
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  rollout call  101/101 (frames 101–101/101)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 4
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  rollout call  101/101 (frames 101–101/101)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 5
━━━━━━━━

# Save to binary

In [ ]:
# save predicted trajectories to disk
save_dir = Path(f'{PROJECT_ROOT}/experiments/turbulent_radiative_layer_2D/results')
save_dir.mkdir(exist_ok=True)

pred_trajs_stacked = torch.cat(pred_trajs, dim=0)  # (N_TRAJ, N_ENSEMBLE, N_ROLLOUT_STEPS, H, W, C)
save_path = save_dir / f"{ckpt_base_name}_rollout_nsampling_steps{NUM_SAMPLING_STEPS}_nensemble{N_ENSEMBLE}_traj.pkl"

with open(save_path, "wb") as f:
    pickle.dump(pred_trajs_stacked, f)

# save real trajectories to disk, save only one time!!
#real_trajs_stacked = torch.cat(real_trajs, dim=0)  # (N_TRAJ, N_ROLLOUT_STEPS, H, W, C)
#save_path = save_dir / f"ground_truth_{well_dataset_name}_traj.pkl"

#with open(save_path, "wb") as f:
#    pickle.dump(real_trajs_stacked, f)